# 傅里叶变换基础

**Fourier Transform Basics**

---

本notebook介绍傅里叶变换的基本概念，为学习OTFS（正交时频空）打下数学基础。OTFS是一种新型调制技术，其核心依赖于Zak变换——傅里叶变换的高维扩展。理解傅里叶变换的时域-频域对应关系是掌握OTFS的关键前提。

## 1. 目标 (Objectives)

通过本notebook，您将：

- **理解连续傅里叶变换与离散傅里叶变换的区别**：连续FT用于理论分析，离散FT用于数值计算
- **理解时域与频域的互补关系**：信号在时域和频域中的表示是等价的，无法同时在两个域中获得任意精度（海森堡不确定性原理）
- **为学习OTFS打下数学基础**：OTFS使用Zak变换，是傅里叶变换在二维时频格上的扩展，理解FT是理解OTFS的前提

## 2. 傅里叶变换直观理解 (Conceptual Understanding)

### 核心思想

傅里叶变换的核心理念源自法国数学家约瑟夫·傅里叶的发现：**任何周期信号都可以表示为一系列正弦波的叠加**。更进一步，非周期信号也可以表示为连续频率的正弦波的积分。这一发现的意义深远——它告诉我们，正弦波是自然的基本构建单元，而傅里叶变换正是揭示这一事实的数学工具。

### 时域与频域的互补关系

海森堡不确定性原理不仅适用于量子力学，也适用于信号处理领域。**信号在时域中的持续时间与其在频域中的带宽成反比**。一个持续时间很短的脉冲（如尖锐的click声）在频域中会分布很广；而一个长时间的正弦波在频域中几乎是单色的。这种互补关系意味着：我们无法同时获得时域和频域的无限精度——时间分辨率和频率分辨率之间存在根本性的制约。

### 对通信原理的重要性

在无线通信中，我们需要在有限的带宽内传输尽可能多的信息。理解傅里叶变换帮助我们：1）理解调制技术如何在频域中移动信号；2）理解多径衰落如何影响信号；3）设计有效的滤波器来分离不同频率的信号。**OFDM（正交频分复用）和OTFS都建立在傅里叶变换的基础之上**，前者使用IFFT/FFT进行调制解调，后者使用Zak变换在时频域进行二维处理。

## 3. 代码演示：连续傅里叶变换 (Continuous Fourier Transform)

连续傅里叶变换的定义为：

$$X(f) = \int_{-\infty}^{\infty} x(t) e^{-j2\pi ft} dt$$

下面我们使用`scipy.integrate.quad`计算矩形脉冲的连续傅里叶变换，结果应该是sinc函数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
%matplotlib inline

# Define the rectangular pulse function
# x(t) = 1 for |t| < T/2, 0 otherwise
def rectangular_pulse(t, T=1.0):
    """Rectangular pulse of duration T centered at 0"""
    return np.where(np.abs(t) < T/2, 1.0, 0.0)

# Continuous Fourier Transform of rectangular pulse
# X(f) = T * sinc(pi * f * T) = T * sin(pi * f * T) / (pi * f * T)
def rect_pulse_ft(f, T=1.0):
    """Analytical FT of rectangular pulse: T * sinc(T * f)"""
    # Handle the DC component (f = 0) separately to avoid division by zero
    if np.abs(f) < 1e-10:
        return T
    return T * np.sin(np.pi * f * T) / (np.pi * f * T)

# Numerical FT using scipy.integrate.quad
def numerical_ft(f, T=1.0):
    """Numerically compute FT of rectangular pulse"""
    def integrand(t):
        return rectangular_pulse(t, T) * np.exp(-1j * 2 * np.pi * f * t)
    # Integrate from -5*T to 5*T (practically covers the pulse)
    real_part, _ = quad(lambda t: np.real(integrand(t)), -5*T, 5*T)
    imag_part, _ = quad(lambda t: np.imag(integrand(t)), -5*T, 5*T)
    return real_part + 1j * imag_part

# Test: compare analytical and numerical FT
frequencies = np.linspace(-5, 5, 500)
T = 1.0  # Pulse duration

# Compute analytical FT
analytical_ft = np.array([rect_pulse_ft(f, T) for f in frequencies])

# Compute numerical FT (sampled)
numerical_ft_samples = np.array([numerical_ft(f, T) for f in frequencies])

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot time-domain signal
axes[0].plot(np.linspace(-3, 3, 600), rectangular_pulse(np.linspace(-3, 3, 600), T), 'b-', linewidth=2)
axes[0].set_xlim(-3, 3)
axes[0].set_xlabel('Time t (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('矩形脉冲 (Rectangular Pulse) - 时域', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# Plot frequency-domain magnitude
axes[1].plot(frequencies, np.abs(analytical_ft), 'b-', linewidth=2, label='解析解 (Analytical)')
axes[1].plot(frequencies, np.abs(numerical_ft_samples), 'r--', linewidth=1.5, label='数值解 (Numerical)', alpha=0.7)
axes[1].set_xlabel('Frequency f (Hz)')
axes[1].set_ylabel('|X(f)|')
axes[1].set_title('矩形脉冲的傅里叶变换 - 频域 (sinc函数)', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

print("连续傅里叶变换完成：矩形脉冲 -> sinc函数")
print("观察到：主瓣宽度与脉冲持续时间成反比（时域越窄，频域越宽）")

## 4. 代码演示：离散傅里叶变换 (Discrete Fourier Transform - DFT)

离散傅里叶变换是对连续FT的数值近似：

$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j2\pi kn/N}, \quad k = 0, 1, \ldots, N-1$$

DFT将N点时域采样转换为N点频域表示。频率轴的缩放规则：
- 频率分辨率：$\Delta f = \frac{f_s}{N}$（采样率/点数）
- 频率范围：$[0, f_s)$ 或 $[-f_s/2, f_s/2]$（使用fftshift）

In [ ]:
# Generate a signal: sum of two sinusoids
# Signal: x[n] = sin(2*pi*f1*t) + 0.5*sin(2*pi*f2*t)
fs = 1000  # Sampling frequency (Hz)
duration = 0.1  # Signal duration (s)
N = 1024  # Number of samples

t = np.arange(N) / fs  # Time vector
f1 = 50   # First frequency component (Hz)
f2 = 150  # Second frequency component (Hz)

# Create composite signal
x = np.sin(2 * np.pi * f1 * t) + 0.5 * np.sin(2 * np.pi * f2 * t)

# Compute DFT using numpy.fft
X = np.fft.fft(x)

# Frequency axis scaling
# For proper frequency scaling, we need frequency resolution = fs/N
freq_resolution = fs / N
frequencies = np.arange(N) * freq_resolution

# Use fftshift to center zero frequency
X_shifted = np.fft.fftshift(X)
frequencies_shifted = np.arange(N) * freq_resolution - fs/2

# Compute magnitude spectrum (only positive frequencies for clarity)
magnitude = np.abs(X_shifted) / N  # Normalize by N

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Time-domain signal
axes[0].plot(t * 1000, x, 'b-', linewidth=1.5)  # Convert to ms
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('复合信号：50Hz + 150Hz 正弦波叠加 - 时域', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, duration * 1000)

# Frequency-domain magnitude
axes[1].plot(frequencies_shifted, magnitude, 'b-', linewidth=1.5)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_title('离散傅里叶变换结果 - 频域 (DFT)', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

# Mark the frequency components
axes[1].axvline(x=f1, color='r', linestyle='--', alpha=0.5, label=f'f1 = {f1} Hz')
axes[1].axvline(x=f2, color='g', linestyle='--', alpha=0.5, label=f'f2 = {f2} Hz')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n离散傅里叶变换参数：")
print(f"  - 采样率 fs = {fs} Hz")
print(f"  - 点数 N = {N}")
print(f"  - 频率分辨率 Δf = {freq_resolution} Hz")
print(f"\n观察到：DFT在{f1}Hz和{f2}Hz处显示两个峰值，对应原始信号的两个频率成分")

## 5. 可视化：时域 ↔ 频域转换 (Time-Frequency Duality)

下面我们创建一个调制的Gaussian脉冲来直观展示时域-频域的对偶性。这是通信系统中常见的信号形式。

In [ ]:
# Create a modulated Gaussian pulse
# Signal: x(t) = Gaussian * cos(2*pi*fc*t)
# This is a typical bandpass signal used in communication systems

# Parameters
fc = 200  # Carrier frequency (Hz)
sigma = 0.02  # Gaussian width (s)
t_center = 0  # Center time (s)

# Time vector
t = np.linspace(-0.1, 0.1, 2000)

# Gaussian envelope
gaussian = np.exp(-((t - t_center) ** 2) / (2 * sigma ** 2))

# Modulated signal: Gaussian pulse with carrier
x = gaussian * np.cos(2 * np.pi * fc * t)

# Compute FT using FFT (efficient DFT implementation)
X = np.fft.fft(x)
X_shifted = np.fft.fftshift(X)

# Frequency axis
fs = 20000  # High sampling rate for good frequency resolution
freq_resolution = fs / len(x)
frequencies = np.arange(len(x)) * freq_resolution - fs/2

# Normalize magnitude
magnitude = np.abs(X_shifted) / len(x)

# Create figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top subplot: Time domain
axes[0].plot(t * 1000, x, 'b-', linewidth=1.5)
axes[0].plot(t * 1000, gaussian, 'r--', linewidth=1, alpha=0.7, label='Gaussian包络')
axes[0].plot(t * 1000, -gaussian, 'r--', linewidth=1, alpha=0.7)
axes[0].set_xlabel('Time (ms)', fontsize=12)
axes[0].set_ylabel('Amplitude', fontsize=12)
axes[0].set_title('调制Gaussian脉冲 - 时域\n(Gaussian envelope * Carrier)', fontsize=14)
axes[0].legend(['信号', '包络'], loc='upper right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-50, 50)

# Bottom subplot: Frequency domain
axes[1].plot(frequencies / 1000, magnitude * 2, 'b-', linewidth=1.5)  # Scale for proper magnitude
axes[1].set_xlabel('Frequency (kHz)', fontsize=12)
axes[1].set_ylabel('Magnitude', fontsize=12)
axes[1].set_title('傅里叶变换结果 - 频域\n(调制将信号搬移到载波频率附近)', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=fc/1000, color='r', linestyle='--', alpha=0.7, label=f'fc = {fc} Hz')
axes[1].axvline(x=-fc/1000, color='r', linestyle='--', alpha=0.7)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n时域-频域对偶性观察：")
print("  - 时域：Gaussian包络限制信号持续时间")
print("  - 频域：调制将频谱中心搬到载波频率fc附近")
print("  - Gaussian包络越宽，频域主瓣越窄（不确定性原理）")

## 6. FFT简介 (Fast Fourier Transform Introduction)

FFT（快速傅里叶变换）是DFT的高效实现。原始DFT的时间复杂度为$O(N^2)$，而FFT将复杂度降低到$O(N \log N)$。

**Cooley-Tukey算法**是最著名的FFT算法，采用分治策略：
1. 将N点DFT分解为两个N/2点DFT（偶索引和奇索引）
2. 递归地分解直到2点DFT
3. 通过蝴蝶操作重组结果

这一突破使得实时信号处理成为可能。

In [ ]:
import time

# Compare manual DFT vs FFT (np.fft.fft) performance

def manual_dft(x):
    """Manual N-point DFT implementation - O(N^2) complexity"""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    twiddle = np.exp(-2j * np.pi * k * n / N)
    return np.dot(twiddle, x)

# Test with different sizes
sizes = [256, 512, 1024, 2048, 4096]
times_manual = []
times_fft = []

print("性能对比：手动DFT vs FFT (numpy)")
print("-" * 50)

for N in sizes:
    # Generate random signal
    x = np.random.randn(N) + 1j * np.random.randn(N)
    
    # Measure manual DFT time
    start = time.time()
    X_manual = manual_dft(x)
    times_manual.append(time.time() - start)
    
    # Measure FFT time
    start = time.time()
    X_fft = np.fft.fft(x)
    times_fft.append(time.time() - start)
    
    # Verify results are equivalent
    error = np.max(np.abs(X_manual - X_fft))
    print(f"N={N:4d}: 手动DFT={times_manual[-1]*1000:8.2f}ms, FFT={times_fft[-1]*1000:8.2f}ms, 误差={error:.2e}")

print("-" * 50)
print("\n结论：FFT相比手动DFT在N=4096时快约")
print(f"{(times_manual[-1]/times_fft[-1]):.1f}x，对于实时信号处理至关重要")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, [t*1000 for t in times_manual], 'b-o', linewidth=2, markersize=8, label='手动DFT O(N^2)')
ax.plot(sizes, [t*1000 for t in times_fft], 'r-s', linewidth=2, markersize=8, label='FFT O(N log N)')
ax.set_xlabel('N (样本数)', fontsize=12)
ax.set_ylabel('执行时间 (ms)', fontsize=12)
ax.set_title('DFT vs FFT 性能对比', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')
plt.show()

## 7. 关联OTFS (Connection to OTFS)

### OFDM与FFT

OFDM（正交频分复用）是当前4G/5G系统的核心技术。在OFDM中：
- **调制**：使用IFFT将并行数据符号映射到正交子载波
- **解调**：使用FFT恢复原始数据

IFFT/FFT的快速计算使得OFDM系统能够使用大量子载波（如5G中通常使用2048或4096点FFT）。

### OTFS与Zak变换

OTFS（正交时频空）是一种新兴的多载波调制方案，核心区别于OFDM：

| 特性 | OFDM | OTFS |
|------|------|------|
| 域 | 时频域 | 时延-多普勒域 |
| 变换 | FFT/IFFT | Zak Transform |
| 处理 | 二维FFT | 二维Zak Transform |
| 信道表示 | 时变OFDM信道 | 时不变等价信道 |
| 优势 | 实现简单 | 对多径衰落更鲁棒 |

**关键洞察**：
- Zak变换本质上是傅里叶变换在高维格点上的扩展
- OTFS将时频域信号通过二维傅里叶相关变换映射到时延-多普勒域
- 理解傅里叶变换的物理意义（时域↔频域对应）是理解OTFS的基础

在后续的OTFS notebook中，我们将详细讨论Zak变换及其与傅里叶变换的关系。

## 8. 思考题 (Review Questions)

1. **时域-频域对偶性**：如果一个信号在时域中是有限长度的，它的频域表示一定是无限延伸的。这种说法正确吗？为什么？

2. **采样定理**：给定采样率$f_s$，一个带限信号的最高可表示频率是多少？若要无失真地恢复原始信号，采样率需要满足什么条件（奈奎斯特率）？

3. **FFT复杂度**：对于N=8192点的变换，FFT相比手动DFT大约快多少倍？假设FFT复杂度为$O(N \log N)$，手动DFT复杂度为$O(N^2)$。

4. **OFDM vs OTFS**：解释为什么OTFS在高速移动场景下比OFDM更能抵抗多普勒频移？提示：考虑两个域中信道的时间不变性。

5. **矩形脉冲的FT**：矩形脉冲（持续时间T）的傅里叶变换是sinc函数。请问sinc函数的零点位置与脉冲持续时间T有什么关系？这如何体现时域-频域的制约关系？

---

**参考答案思路**：
1. 正确。根据不确定性原理，时域有限意味着频域无限（但能量集中）
2. 最高频率 = fs/2（奈奎斯特频率）；fs ≥ 2*B（B为信号带宽）
3. 约8192/log₂(8192) ≈ 8192/13 ≈ 630倍
4. OTFS在时延-多普勒域中信道是时不变的，而OFDM在时频域中信道随时间快速变化
5. sinc零点位置 = ±1/T, ±2/T, ... 时域越窄（T小），频域主瓣越宽

## 总结 (Summary)

本notebook介绍了傅里叶变换的基础知识：

- **连续FT**用于理论分析，定义积分形式的时域↔频域变换
- **离散FT (DFT)**是连续FT的数值近似，用于实际计算
- **FFT**是DFT的高效实现，使实时信号处理成为可能
- **时域-频域对偶性**揭示了信号在两个域中的互补表示
- **OFDM**使用IFFT/FFT，**OTFS**使用Zak变换（傅里叶变换的扩展）

这些概念为学习OTFS提供了必要的数学基础。